## Notebook 13 — Thread Count Latency Analysis
**Project:** Machine Learning for High Performance Optical Sorting
**Author:** Mohamed Tawfeek
**Description:** RF inference latency benchmarked across thread configurations (n_jobs), with MLP as reference and comparison against a 50 ms industrial target


In [1]:
import os
import sys
import time
import json
import pickle
import platform
import subprocess
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from PIL import Image
from skimage.feature import local_binary_pattern

import sys
sys.path.insert(0, os.path.abspath('..'))

In [2]:
RESULTS_DIR = os.path.join("..", "results")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
FEATURES_PATH = os.path.join(RESULTS_DIR, "features", "features.npy")
LABELS_PATH   = os.path.join(RESULTS_DIR, "features", "labels.npy")  
RF_MODEL_PATH = os.path.join(RESULTS_DIR, "rf_final_model.pkl")
MLP_MODEL_PATH = os.path.join(RESULTS_DIR, "mlp_final_model.pkl")
MLP_SCALER_PATH = os.path.join(RESULTS_DIR, "mlp_scaler.pkl")

In [3]:
# Benchmarking parameters: number of timing repetitions per measurement, RF thread configurations to evaluate, and the 50 ms industrial latency target

N_TIMING_RUNS = 100    
N_JOBS_LIST   = [1, 2, 4, -1]
LATENCY_TARGET_MS = 50.0  

In [4]:
print("Loading features...")
all_features = np.load(FEATURES_PATH)
all_labels   = np.load(LABELS_PATH)
 
X_train_val, X_test, y_train_val, y_test = train_test_split(
    all_features, all_labels,
    test_size=0.15,
    random_state=42,
    stratify=all_labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.176,
    random_state=42,
    stratify=y_train_val
)
 
print(f"Test set: {X_test.shape[0]} images, {X_test.shape[1]} features")

Loading features...
Test set: 380 images, 122 features


In [5]:
with open(MLP_SCALER_PATH, "rb") as f:
    scaler = pickle.load(f)
X_test_scaled = scaler.transform(X_test)

In [6]:
with open(RF_MODEL_PATH, "rb") as f:
    rf_base = pickle.load(f)

with open(MLP_MODEL_PATH, "rb") as f:
    mlp = pickle.load(f)

In [7]:
# Defines the feature extraction pipeline, selects one representative image from the dataset, and measures extraction latency as the shared first-stage cost for all configurations

IMAGE_SIZE     = (224, 224)
HISTOGRAM_BINS = 32
DATASET_PATH   = os.path.join("..", "datasets", "dataset-resized")
CLASSES        = ["glass", "paper", "cardboard", "plastic", "metal", "trash"]
 
from src.features import extract_features_from_image as extract_features
 
sample_img_path = None
for cls in CLASSES:
    cls_dir = os.path.join(DATASET_PATH, cls)
    for f in sorted(os.listdir(cls_dir)):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            sample_img_path = os.path.join(cls_dir, f)
            break
    if sample_img_path:
        break
 
if sample_img_path is None:
    print("ERROR: Could not locate any image in dataset. Check DATASET_PATH.")
    sys.exit(1)
 
sample_image = Image.open(sample_img_path).convert("RGB")
 
print(f"\nMeasuring feature extraction latency ({N_TIMING_RUNS} runs)...")
fe_times = []
for _ in range(N_TIMING_RUNS):
    t0 = time.perf_counter()
    _ = extract_features(sample_image)
    fe_times.append((time.perf_counter() - t0) * 1000)
 
fe_mean = np.mean(fe_times)
fe_std  = np.std(fe_times)
print(f"  Feature extraction: {fe_mean:.3f} ± {fe_std:.3f} ms")


Measuring feature extraction latency (100 runs)...


  Feature extraction: 18.376 ± 0.417 ms


In [8]:
# Clones the trained RF with each n_jobs setting by transferring estimators directly without retraining, then measures single-image classification latency per configuration

print(f"\nMeasuring RF classification latency at each n_jobs...")
 
rf_results = {}
single_image_feature = X_test[[0]]  
 
for n_jobs in N_JOBS_LIST:
    rf_variant = RandomForestClassifier(
        n_estimators=rf_base.n_estimators,
        max_depth=rf_base.max_depth,
        random_state=42,
        n_jobs=n_jobs
    )
    rf_variant.estimators_ = rf_base.estimators_
    rf_variant.classes_     = rf_base.classes_
    rf_variant.n_classes_   = rf_base.n_classes_
    rf_variant.n_features_in_ = rf_base.n_features_in_
    rf_variant.n_outputs_   = rf_base.n_outputs_
 
    label = "all cores" if n_jobs == -1 else f"{n_jobs} thread{'s' if n_jobs > 1 else ''}"
    print(f"  n_jobs={n_jobs} ({label}): ", end="", flush=True)
 
    _ = rf_variant.predict(single_image_feature)
 
    clf_times = []
    for _ in range(N_TIMING_RUNS):
        t0 = time.perf_counter()
        rf_variant.predict(single_image_feature)
        clf_times.append((time.perf_counter() - t0) * 1000)
 
    clf_mean = np.mean(clf_times)
    clf_std  = np.std(clf_times)
    total    = fe_mean + clf_mean
    print(f"{clf_mean:.3f} ± {clf_std:.3f} ms  →  total {total:.3f} ms")
 
    rf_results[str(n_jobs)] = {
        "n_jobs": n_jobs,
        "label": label,
        "clf_mean_ms": clf_mean,
        "clf_std_ms": clf_std,
        "total_mean_ms": total,
        "meets_50ms_target": total <= LATENCY_TARGET_MS
    }


Measuring RF classification latency at each n_jobs...
  n_jobs=1 (1 thread): 

9.293 ± 0.457 ms  →  total 27.668 ms
  n_jobs=2 (2 threads): 

36.305 ± 15.741 ms  →  total 54.681 ms
  n_jobs=4 (4 threads): 

35.122 ± 13.517 ms  →  total 53.497 ms
  n_jobs=-1 (all cores): 

39.414 ± 17.928 ms  →  total 57.789 ms


In [9]:
# Measures MLP single-image classification latency as a reference point for comparison against the RF thread configurations

print(f"\nMeasuring MLP classification latency ({N_TIMING_RUNS} runs)...")
single_image_scaled = X_test_scaled[[0]]
 
_ = mlp.predict(single_image_scaled)   # warm-up
 
mlp_times = []
for _ in range(N_TIMING_RUNS):
    t0 = time.perf_counter()
    mlp.predict(single_image_scaled)
    mlp_times.append((time.perf_counter() - t0) * 1000)
 
mlp_clf_mean = np.mean(mlp_times)
mlp_clf_std  = np.std(mlp_times)
mlp_total    = fe_mean + mlp_clf_mean
print(f"  MLP: {mlp_clf_mean:.3f} ± {mlp_clf_std:.3f} ms  →  total {mlp_total:.3f} ms")


Measuring MLP classification latency (100 runs)...
  MLP: 0.120 ± 0.019 ms  →  total 18.496 ms


In [10]:
output_data = {
    "feature_extraction": {
        "mean_ms": fe_mean,
        "std_ms": fe_std,
        "n_runs": N_TIMING_RUNS
    },
    "rf_by_n_jobs": rf_results,
    "mlp": {
        "clf_mean_ms": mlp_clf_mean,
        "clf_std_ms": mlp_clf_std,
        "total_mean_ms": mlp_total,
        "meets_50ms_target": mlp_total <= LATENCY_TARGET_MS
    },
    "latency_target_ms": LATENCY_TARGET_MS,
}
 
json_path = os.path.join(RESULTS_DIR, "thread_count_results.json")
with open(json_path, "w") as f:
    json.dump(output_data, f, indent=2, default=str)
print(f"\nSaved: {json_path}")


Saved: ..\results\thread_count_results.json


In [11]:
# Stacked bar chart of end-to-end latency (extraction + classification) for each RF thread count with MLP as reference, and the 50 ms industrial target line annotated

fig, ax = plt.subplots(figsize=(8, 5))
 
n_jobs_labels = []
clf_means     = []
clf_stds      = []
totals        = []
 
for n_jobs in N_JOBS_LIST:
    key = str(n_jobs)
    label = "n_jobs=−1\n(all cores)" if n_jobs == -1 else f"n_jobs={n_jobs}"
    n_jobs_labels.append(label)
    clf_means.append(rf_results[key]["clf_mean_ms"])
    clf_stds.append(rf_results[key]["clf_std_ms"])
    totals.append(rf_results[key]["total_mean_ms"])
 
x = np.arange(len(N_JOBS_LIST))
bar_width = 0.35
 
bars_fe  = ax.bar(x, [fe_mean] * len(N_JOBS_LIST), bar_width,
                  label="Feature extraction (shared)", color="#4878D0", alpha=0.85)
bars_clf = ax.bar(x, clf_means, bar_width, bottom=[fe_mean] * len(N_JOBS_LIST),
                  label="RF classification", color="#EE854A", alpha=0.85,
                  yerr=clf_stds, error_kw={"elinewidth": 1.2, "capsize": 4})
 
mlp_x = len(N_JOBS_LIST) + 0.3
ax.bar(mlp_x, fe_mean, bar_width, color="#4878D0", alpha=0.85)
ax.bar(mlp_x, mlp_clf_mean, bar_width, bottom=fe_mean, color="#6ACC65", alpha=0.85,
       label="MLP classification", yerr=mlp_clf_std,
       error_kw={"elinewidth": 1.2, "capsize": 4})
 
ax.axhline(LATENCY_TARGET_MS, color="red", linestyle="--", linewidth=1.5,
           label=f"50 ms target")
 
for i, total in enumerate(totals):
    ax.text(x[i], total + 1.0, f"{total:.1f}", ha="center", va="bottom",
            fontsize=8.5, color="#333333")
ax.text(mlp_x, mlp_total + 1.0, f"{mlp_total:.1f}", ha="center", va="bottom",
        fontsize=8.5, color="#333333")
 
all_x_labels = n_jobs_labels + ["MLP\n(reference)"]
ax.set_xticks(list(x) + [mlp_x])
ax.set_xticklabels(all_x_labels, fontsize=9)
ax.set_ylabel("End-to-end latency (ms)", fontsize=10)
ax.set_xlabel("RF thread configuration / model", fontsize=10)
ax.set_title("End-to-end latency vs RF thread count\n(feature extraction + classification, single-image inference)", fontsize=10)
ax.legend(fontsize=8.5, loc="upper right")
ax.set_ylim(0, max(max(totals), mlp_total) * 1.25)
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.grid(axis="y", linestyle=":", alpha=0.5)
 
plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "thread_count_latency.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {fig_path}")

Saved: ..\results\figures\thread_count_latency.png


In [12]:
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Feature extraction:  {fe_mean:.3f} ± {fe_std:.3f} ms")
print(f"MLP classification:  {mlp_clf_mean:.3f} ± {mlp_clf_std:.3f} ms  →  total {mlp_total:.3f} ms")
print()
print(f"{'n_jobs':<12} {'Clf (ms)':<16} {'Total (ms)':<16} {'≤50ms?'}")
print("-" * 56)
for n_jobs in N_JOBS_LIST:
    key = str(n_jobs)
    r = rf_results[key]
    meets = "YES" if r["meets_50ms_target"] else "NO"
    print(f"{r['label']:<12} {r['clf_mean_ms']:<16.3f} {r['total_mean_ms']:<16.3f} {meets}")
print("=" * 60)


SUMMARY
Feature extraction:  18.376 ± 0.417 ms
MLP classification:  0.120 ± 0.019 ms  →  total 18.496 ms

n_jobs       Clf (ms)         Total (ms)       ≤50ms?
--------------------------------------------------------
1 thread     9.293            27.668           YES
2 threads    36.305           54.681           NO
4 threads    35.122           53.497           NO
all cores    39.414           57.789           NO
